# Debug a Broken Training Pipeline — Solution

**Difficulty**: 🟡 Medium

### Problem Statement

*"The model trains, but something is off."* Research-engineer and research-scientist interviews often hand you a hundred lines of training code and ask you to find what is wrong. Nothing crashes. The bugs are **silent** and show up only as symptoms: a loss that stalls at a suspicious value, an accuracy that changes when you evaluate the same model twice, memory that grows every epoch, a model that trains fine yet scores 0%. What is being tested is diagnostic reasoning, going from **symptom → hypothesis → targeted test → fix**. Recall of APIs matters much less.

Below is a complete, small image-classification pipeline on a synthetic dataset: 16×16 RGB "photos" of four procedurally drawn shapes, stored as `uint8` in HWC layout like images decoded by PIL/OpenCV. Nothing is downloaded. The pipeline runs end to end without errors and contains **exactly 9 bugs** spread over its six functions. They are not spread evenly: a function may have several bugs, or none.

---

### Requirements

Fix the six functions so each does what its docstring says (keep the signatures):

1. **`to_tensor(images)`**: `uint8 (N, H, W, C)` → `float32 (N, C, H, W)` in $[0, 1]$, with `out[n, c, h, w] = images[n, h, w, c] / 255`.
2. **`compute_normalization_stats(images)`**: float `(N, C, H, W)` → `(mean, std)`, each of shape `(C,)`, i.e. statistics of each channel over all of $N$, $H$, $W$.
3. **`normalize(images, mean, std)`**: per-channel $(x - \mu_c)/\sigma_c$.
4. **`make_model(num_classes)`**: a CNN mapping `(B, 3, 16, 16)` → `(B, num_classes)` **logits** for `nn.CrossEntropyLoss`.
5. **`train_one_epoch(model, loader, optimizer, loss_fn) -> float`**: one pass over `loader` with a standard training step per batch. The update for a batch must depend only on that batch, and dropout/BatchNorm must behave as in training. Returns the mean of the per-batch losses as a Python `float`.
6. **`evaluate(model, loader) -> float`**: accuracy in $[0, 1]$ over all samples of `loader`, with dropout/BatchNorm in their inference behaviour and no memory wasted on things inference does not need. Evaluating the same model twice must give the same number, and evaluation must not change the model.

Once fixed, the driver cell reaches **> 95% test accuracy in 5 epochs** on CPU in a few seconds.

---

### Constraints

- ✅ Fix the bugs in place with minimal diffs, as you would in a code review, and note for each one the **symptom** it causes.
- ❌ Do not change the dataset generator, the driver cell, the optimizer or the hyper-parameters.
- ✅ Try to find all 9 from the symptoms and by reading the code **before** running `check(...)`. You won't have a grader in an interview. The grader tests each function's behaviour separately, so it then points at whatever is left.

---

<details>
  <summary>💡 Hint</summary>

  - **Look at the data after every transform.** The ASCII viewer below prints an image. Does it still look like the shape it is labelled as?
  - After normalisation, each channel of the *training* set should have mean ≈ 0 and std ≈ 1. Print it.
  - **Know your reference numbers.** The initial loss should be ≈ $\log C$. Is there a floor the loss cannot go below? What would cause one?
  - **Evaluate the same model twice.** Different answers mean something random is still switched on.
  - Check the *type* of what the training loop returns. Does it have a `.grad_fn`?
  - For each line of each loop, ask: what state does this depend on (`.grad`, `model.training`, the autograd graph), and who resets it?

</details>

---

In [ ]:
import math

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
NUM_CLASSES = 4
CLASS_NAMES = ["horizontal bar", "vertical bar", "hollow square", "diagonal"]


def make_shapes_dataset(n, seed, size=16):
    # Synthetic "photos": uint8 (n, size, size, 3) in HWC layout, like decoded JPEGs. Labels (n,).
    # This generator is correct -- the bugs are elsewhere.
    g = torch.Generator().manual_seed(seed)
    labels = torch.randint(0, NUM_CLASSES, (n,), generator=g)
    background = torch.tensor([0.15, 0.45, 0.70])                 # each channel has its own level
    imgs = background + 0.08 * torch.randn(n, size, size, 3, generator=g)
    yy = torch.arange(size).view(1, size, 1)
    xx = torch.arange(size).view(1, 1, size)
    r = torch.randint(4, size - 4, (n, 1, 1), generator=g)        # random shape centre
    c = torch.randint(4, size - 4, (n, 1, 1), generator=g)
    dy, dx = yy - r, xx - c
    shapes = torch.stack([
        (dy.abs() <= 1) & (dx.abs() <= 4),                                          # horizontal bar
        (dx.abs() <= 1) & (dy.abs() <= 4),                                          # vertical bar
        (dy.abs() <= 3) & (dx.abs() <= 3) & ~((dy.abs() <= 1) & (dx.abs() <= 1)),   # hollow square
        ((dy - dx).abs() <= 1) & (dy.abs() <= 4),                                   # diagonal
    ], dim=1)                                                     # (n, 4, size, size)
    mask = shapes[torch.arange(n), labels]                        # (n, size, size)
    color = (1 - background) + 0.15 * torch.randn(n, 1, 1, 3, generator=g)
    imgs = torch.where(mask[..., None], color, imgs)
    return (imgs.clamp(0, 1) * 255).round().to(torch.uint8), labels


train_images, train_labels = make_shapes_dataset(1024, seed=0)
test_images, test_labels = make_shapes_dataset(512, seed=1)
print("train:", tuple(train_images.shape), train_images.dtype, " test:", tuple(test_images.shape))
print("class counts:", torch.bincount(train_labels).tolist())

In [ ]:
# Debugging toolkit (correct, use it freely)
def ascii_image(gray_hw):
    # Print a (H, W) image as ASCII art, min-max scaled.
    ramp = " .:-=+*#%@"
    g = gray_hw.float()
    g = (g - g.min()) / (g.max() - g.min() + 1e-8)
    for row in g:
        print(" ".join(ramp[int(v * (len(ramp) - 1))] for v in row))


print(f"train_images[0] is a {CLASS_NAMES[train_labels[0]]}. Its red channel (raw HWC):")
ascii_image(train_images[0, :, :, 0])

### The pipeline (fixed: each fix is marked `# FIX n`)

In [ ]:
def to_tensor(images):
    """uint8 (N, H, W, C) images in HWC layout -> float32 (N, C, H, W) in [0, 1]."""
    # FIX 1: reshape() only reinterprets the memory order -- it never moves axes. HWC -> CHW
    # needs a permute (the old code produced a fixed scramble of pixels and channels).
    return images.permute(0, 3, 1, 2).float() / 255.0


def compute_normalization_stats(images):
    """Per-channel mean and std of a float (N, C, H, W) batch -> two tensors of shape (C,)."""
    # FIX 2: images.reshape(C, -1) groups consecutive MEMORY, i.e. blocks of whole samples,
    # not channels -- every "channel" got roughly the global mean. Reduce over N, H, W instead
    # (equivalently: images.transpose(0, 1).reshape(C, -1)).
    return images.mean(dim=(0, 2, 3)), images.std(dim=(0, 2, 3))


def normalize(images, mean, std):
    """Standardise each channel: (x - mean[c]) / std[c]."""
    # (no bug here)
    return (images - mean.view(1, -1, 1, 1)) / std.view(1, -1, 1, 1)

In [ ]:
def make_model(num_classes=NUM_CLASSES):
    """Small CNN: (B, 3, 16, 16) -> (B, num_classes) logits for nn.CrossEntropyLoss."""
    return nn.Sequential(
        nn.Conv2d(3, 16, kernel_size=3, padding=1), nn.BatchNorm2d(16), nn.ReLU(),
        nn.MaxPool2d(2),                                          # 16x16 -> 8x8
        nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
        nn.MaxPool2d(2),                                          # 8x8 -> 4x4
        nn.Flatten(),
        nn.Dropout(0.3),
        nn.Linear(32 * 4 * 4, num_classes),
        # FIX 3: no Softmax -- nn.CrossEntropyLoss applies log-softmax itself. Softmax first
        # squashes the "logits" into [0, 1], so the loss can never go below log(1 + (C-1)/e).
    )

In [ ]:
def train_one_epoch(model, loader, optimizer, loss_fn):
    """One pass over `loader`, one optimizer step per batch. Returns the mean batch loss (float)."""
    model.train()                       # FIX 4: evaluate() left the model in eval mode
    total_loss = 0.0
    for images, labels in loader:
        optimizer.zero_grad()           # FIX 5: .grad accumulates across backward() calls
        logits = model(images)
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()       # FIX 6: `+= loss` kept every batch's graph alive
    return total_loss / len(loader)


@torch.no_grad()                        # FIX 8: no autograd graph during evaluation
def evaluate(model, loader):
    """Accuracy in [0, 1] of `model` over every sample in `loader`."""
    model.eval()                        # FIX 7: dropout off, BatchNorm uses running stats
    correct, total = 0, 0
    for images, labels in loader:
        logits = model(images)
        preds = logits.argmax(dim=1)    # FIX 9: max(dim=1)[0] is the max VALUE, not the class
        correct += (preds == labels).sum().item()
        total += labels.numel()
    return correct / total

### Run it

In [ ]:
# The driver: do not modify. Run it, read the output, form hypotheses.
torch.manual_seed(0)
train_x, test_x = to_tensor(train_images), to_tensor(test_images)
print(f"after to_tensor: {tuple(train_x.shape)}. Red channel of the same {CLASS_NAMES[train_labels[0]]}:")
ascii_image(train_x[0, 0])

mean, std = compute_normalization_stats(train_x)
train_x, test_x = normalize(train_x, mean, std), normalize(test_x, mean, std)
print("normalised train set  per-channel mean:", [round(v, 3) for v in train_x.mean(dim=(0, 2, 3)).tolist()],
      " std:", [round(v, 3) for v in train_x.std(dim=(0, 2, 3)).tolist()])

train_loader = DataLoader(TensorDataset(train_x, train_labels), batch_size=64, shuffle=True)
test_loader = DataLoader(TensorDataset(test_x, test_labels), batch_size=256)
model = make_model(NUM_CLASSES)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)
loss_fn = nn.CrossEntropyLoss()
print(f"reference: chance-level loss log(C) = {math.log(NUM_CLASSES):.3f}")

history = []
for epoch in range(5):
    train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn)
    acc = evaluate(model, test_loader)
    acc_again = evaluate(model, test_loader)            # same model, same data
    history.append(train_loss)
    print(f"epoch {epoch}: train loss {float(train_loss):.4f} [{type(train_loss).__name__}]  "
          f"test acc {acc:.3f}  (evaluated again: {acc_again:.3f})")

In [ ]:
# What a healthy pipeline looks like (these pass once all 9 bugs are fixed)
assert isinstance(history[-1], float), \
    f"train_one_epoch should return a plain float, got {type(history[-1]).__name__}"
assert acc == acc_again, "evaluating the same model twice must give the same accuracy"
norm_mean = train_x.mean(dim=(0, 2, 3))
assert torch.allclose(norm_mean, torch.zeros(3), atol=1e-3), \
    f"normalised training data should have per-channel mean 0, got {norm_mean.tolist()}"
assert history[-1] < 0.3, f"final train loss {history[-1]:.3f} -- this task is easy"
assert acc > 0.95, f"test accuracy {acc:.3f} -- a healthy pipeline gets > 95% here"
print("pipeline looks healthy")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("debug-training", to_tensor=to_tensor, compute_normalization_stats=compute_normalization_stats,
      normalize=normalize, make_model=make_model, train_one_epoch=train_one_epoch, evaluate=evaluate)

### The 9 bugs: symptom → detection (each fix is marked `# FIX n` in the code above)

Single-bug numbers are from runs of this exact driver with only that bug present: 5 epochs, CPU, seeds as above. Your numbers may differ slightly. The healthy pipeline ends at train loss ≈ 0.02 and test accuracy ≈ 0.99–1.00.

| # | Where | Bug | Symptom (only this bug present) | How you'd detect it |
|---|---|---|---|---|
| 1 | `to_tensor` | `reshape(n, c, h, w)` instead of `permute(0, 3, 1, 2)` | **Runs, and the model still learns** (≈ 0.96 test acc. vs ≈ 1.00). A reshape is a fixed permutation of the pixels, so the information survives, but every image becomes stripes and locality is destroyed. On real data this costs much more. | **Look at a batch after every transform** (the ASCII view after `to_tensor` is garbage). Unit test: `out[n, c, h, w] == images[n, h, w, c]`. |
| 2 | `compute_normalization_stats` | `images.reshape(C, -1)` groups consecutive memory (blocks of *samples*), not channels | Each "channel" statistic ≈ the global mean/std. The normalised training set has channel means ≈ (−0.9, 0.05, 0.8) and stds ≈ (1.0, 0.4, 0.6) instead of 0 and 1. **Metrics barely move**: the BatchNorm after the first conv absorbs the offset. | Print per-channel mean/std of the normalised training set. Compare with `images.mean(dim=(0, 2, 3))`. |
| 3 | `make_model` | `nn.Softmax` before `nn.CrossEntropyLoss` | Train loss **plateaus around 0.75**. The floor is $\log(1 + (C-1)/e) = 0.744$ for $C = 4$, because "logits" in $[0,1]$ cannot express confidence. Gradients are tiny once the softmax saturates, so it learns slowly, although accuracy can still end up high. | A loss stuck at a "weird constant". Outputs that sum to 1. Try to overfit a single batch: the loss won't go to 0. |
| 4 | `train_one_epoch` | no `model.train()` | Epoch 0 trains normally. From epoch 1 on, the model trains **in eval mode** because `evaluate` switched it. Dropout is off and BN is frozen at the epoch-0 running stats, so the train loss drops *faster* (≈ 0.006 vs 0.02). Harmless on this toy, but on real data it means no regularisation and stale BN statistics. | Assert `model.training` inside the loop, or register a forward hook. Watch the train loss change character right after the first evaluation. |
| 5 | `train_one_epoch` | no `optimizer.zero_grad()` | `.grad` keeps the **sum of all past gradients**, so each step follows a stale, growing direction. Train loss *rises* to ≈ 3–5, far above log C = 1.386, and accuracy wanders at 40–60%. | A loss that goes up or oscillates while the LR is sane. Grad norm grows every step. Compare one epoch against manual clean steps on a fixed batch. |
| 6 | `train_one_epoch` | `total_loss += loss` (a tensor) | Metrics identical, but the function returns a `Tensor` with `grad_fn`. `float(...)` on it triggers a *requires_grad* warning, and every batch's graph (with its activations) stays alive. Memory grows with every step, and `history` keeps them alive across epochs, the classic slow OOM. | `type(loss)` / `.grad_fn` on the returned value. GPU memory rising epoch after epoch. Use `loss.item()` (or `.detach()`) for logging. |
| 7 | `evaluate` | no `model.eval()` | **Two evaluations of the same model disagree** (e.g. 0.941 vs 0.945), because dropout is active. BN normalises with test-batch statistics *and updates its running stats with test data*, a silent test-set leak that changes the model. | Evaluate twice. Check that BN buffers are unchanged after evaluation. |
| 8 | `evaluate` | no `torch.no_grad()` | Same accuracy, but every forward pass records a graph and keeps activations for a backward that never comes. Evaluation is slower and uses roughly train-level memory, so large eval batches OOM. | `logits.requires_grad` is True inside evaluation. Memory profile of evaluation. Use `torch.no_grad()` / `torch.inference_mode()`. |
| 9 | `evaluate` | `logits.max(dim=1)[0]` (the max **value**) compared with integer labels | Test accuracy **exactly 0.000** while the train loss falls normally. (With bug 3 also present, softmax outputs that saturate to exactly 1.0 even match label 1 now and then, so you see a few % instead of 0.) | Accuracy *below chance* (25%) is a red flag for a bookkeeping bug, not a modelling bug. Print a few `preds` next to `labels`. |

**With all nine present** (the question notebook), the train loss hovers at **≈ 1.3–1.4 ≈ log C**: bugs 3 and 5 together stop learning almost completely. Test accuracy is 0–9% and differs between the two calls, and the returned loss is a `Tensor`. Several bugs hide behind others. Bug 4 only matters once bug 7 is fixed (until then nothing ever switches the model to eval mode). Bug 9 makes every accuracy-based symptom of the others invisible.

### Why it works: a debugging method that finds these quickly

1. **Make it observable.** Look at inputs after every transform (shape *and* content), print summary statistics, and log the train loss, the grad norm and the eval metric per epoch.
2. **Know the reference numbers.** Initial loss ≈ $\log C$ for a fresh classifier. Chance accuracy is $1/C$. Accuracy **below** chance means a bookkeeping bug. A loss that plateaus at a constant you can compute means a structural bug (softmax + CE, labels decoupled from images, a frozen layer).
3. **Test determinism.** Evaluate twice. Re-run with the same seed. Anything that changes points at stochastic layers in the wrong mode or uncontrolled randomness.
4. **Isolate state.** Every loop carries hidden state: `.grad` (accumulates), `model.training` (persists across calls), the autograd graph (lives as long as any tensor references it), BN running stats (updated by every train-mode forward). Each of bugs 4–8 is one of these never being reset.
5. **Overfit one batch.** A correct model and loop should drive the loss on a single batch to ≈ 0 within a few hundred steps. Failing that rules out data issues and points at the model, the loss or the loop.
6. **Unit-test each stage against a trusted reference** (exactly what the grader does). Compare with a hand-computed value, a library function, or a "clean" re-implementation on a tiny input.

### Complexity

The fixed CNN costs ≈ 0.4 M multiply-adds per 16×16 image forward (two small convs dominate), so one epoch over 1024 images takes about a second on a CPU. The debugging tools are cheap too: hooks and assertions add negligible overhead, and "evaluate twice" doubles only the evaluation cost.

### Common bugs (beyond the nine planted here)

- **Normalisation statistics from the test set**, or from train+test: a leak that inflates test scores. Compute them on the training split only and reuse them everywhere.
- **Labels shuffled independently of the inputs** (e.g. two `randperm` calls). The loss plateaus at exactly $\log C$ and accuracy stays at chance.
- Training-time augmentation (random crops/flips) applied to the validation set. Augmentations in the wrong order, e.g. normalising before `ToTensor`'s `/255`.
- Float `(N, C)` targets passed to `CrossEntropyLoss` are silently read as class *probabilities*. That is fine for exact one-hot targets, but `ignore_index` no longer applies and class weights are averaged over $N$ instead of $\sum w_{y_i}$. A related bug is `BCEWithLogitsLoss` applied to sigmoid outputs (sigmoid applied twice).
- `scheduler.step()` per batch when it was configured per epoch (or vice versa), or with gradient accumulation stepping the optimizer every micro-batch.
- `model.eval()` forgotten at **inference/export** time. It is the same bug as #7, but it ships to production.
- Creating new tensors inside `forward` without the input's `device`/`dtype`. On GPU this crashes, and under mixed precision a float32 constant silently upcasts every activation it touches.
- Switching a loss from mean to sum reduction (or changing the batch size under a sum). This silently rescales the gradient, so an SGD learning rate tuned for the mean is suddenly far too large. (Adam's update is nearly invariant to a constant gradient scale, but a gradient-clipping threshold is not.)

### Interview follow-ups

1. **The training loss plateaus at exactly 1.386 with 4 classes. What do you suspect?** $\log 4 = 1.386$ is the loss of predicting the uniform distribution: the model learns nothing that depends on the input. Suspect labels decoupled from inputs (a shuffling or indexing bug), inputs that carry no information (all zeros or saturated after a broken preprocessing step), a dead network (all ReLUs dead, zero LR, frozen params), or gradients not reaching the weights (`detach()`, `requires_grad=False`, an optimizer built over the wrong parameters).
2. **How do you check that gradients actually reach every layer?** After one `backward()`, assert that each parameter's `.grad` is not None and has a nonzero norm. Log per-layer gradient and update ratios ($\|\Delta W\|/\|W\| \approx 10^{-3}$ is healthy). Verify that `optimizer.param_groups` contains the same objects as `model.parameters()`. A common bug is creating the optimizer before re-initialising or replacing a layer.
3. **Why does BatchNorm make "train mode vs eval mode" so consequential, and what do you do with tiny per-GPU batches?** In train mode, BN normalises with the current batch's statistics and updates running averages. In eval mode it uses the running averages. With tiny batches, the batch statistics are noisy and the running stats poor. Use SyncBatchNorm, freeze BN (common when fine-tuning detectors), or switch to GroupNorm/LayerNorm, which behave identically in both modes.
4. **Evaluation gives different results on every run even with `model.eval()`. What else can be nondeterministic?** A `DataLoader` with `shuffle=True` together with a metric that depends on order or batching (e.g. a per-batch average with a ragged last batch), random augmentation in the eval transforms, nondeterministic GPU kernels (atomics in scatter/index_add, some cuDNN algorithms; see `torch.use_deterministic_algorithms(True)`). Across machines, TF32/mixed-precision settings also change results.
5. **Memory grows every epoch. How do you find the leak?** Check what keeps tensors alive: losses stored without `.item()`, outputs appended to a list for metrics without `.detach().cpu()`, hooks that capture activations, and evaluation without `no_grad`. Tools: `torch.cuda.memory_allocated()` per step, `torch.cuda.memory_summary()`, the memory snapshot visualiser, and checking which logged tensors have a `grad_fn`.
6. **The model trains to 99% train accuracy but 25% test accuracy on a balanced 4-class task. Is it overfitting?** Not necessarily. Exactly chance level on test points at a **pipeline mismatch** between train and test before any overfitting story: different normalisation or preprocessing (HWC vs CHW, BGR vs RGB, missing `/255`), labels re-mapped differently, BN statistics poisoned by a train-mode evaluation, or an evaluation bug like #9. Evaluate the *training* set through the evaluation code path: if that also gives 25%, the bug is in evaluation or preprocessing, not in generalisation.